<a href="https://colab.research.google.com/github/Emily-als/worklens-llm/blob/feature%2Fexperiment-data-preparation/03_experiment_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# WorkLens - Experiment Data Preparation
## Load Processed Data

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split
from google.colab import drive

SEED = 42

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
DATA_DIR = Path(
    "/content/drive/MyDrive/LLM Project/DATASETS/REAL/Glassdoor/v1"
)

assert DATA_DIR.is_dir(), "Dataset folder not found"

df_train = pd.read_parquet(DATA_DIR / "train.parquet")

print("Training dataset shape:", df_train.shape)
print("Columns:", df_train.columns.tolist())

display(df_train.head())

Training dataset shape: (387057, 14)
Columns: ['review_id', 'firm', 'headline', 'pros', 'cons', 'work_life_balance', 'comp_benefits', 'senior_mgmt', 'work_life_balance_label', 'comp_benefits_label', 'senior_mgmt_label', 'input_text', 'is_mixed', 'label_combination']


,review_id,firm,headline,pros,cons,work_life_balance,comp_benefits,senior_mgmt,work_life_balance_label,comp_benefits_label,senior_mgmt_label,input_text,is_mixed,label_combination
0,198590,EY,Great place for Grads to learn and discover wh...,Amazing knowledge pool\nGood investment in tra...,Typical challenges or a large organisation - h...,3.0,4.0,3.0,Medium,High,Medium,Headline: Great place for Grads to learn and d...,False,Medium_High_Medium
1,65111,Babcock-International-Group,17 years and still here,Great place to have a full and diverse career ...,Communication not the best but getting better\...,2.0,3.0,3.0,Low,Medium,Medium,Headline: 17 years and still here\nPros: Great...,False,Low_Medium_Medium
2,648340,next,"Great benefits, too rigid in terms of procedures","Great benefits, pay even at sales assistant le...","Too strict in terms of procedures, very rigid,...",2.0,4.0,2.0,Low,High,Low,"Headline: Great benefits, too rigid in terms o...",True,Low_High_Low
3,439020,Microsoft,Microsoft veteran - 10 years,It is an awesome company in terms of how it tr...,Not very good at consumer stuff. Re-orgs all t...,5.0,5.0,4.0,High,High,High,Headline: Microsoft veteran - 10 years\nPros: ...,False,High_High_High
4,33142,Apple,Great culture,open-minded environment and a great place to w...,Retail branch working schedule is not a monday...,5.0,5.0,5.0,High,High,High,Headline: Great culture\nPros: open-minded env...,False,High_High_High


In [4]:
## Standardize Dataset Schema

In [5]:
column_mapping = {
    "review_id": "id",
    "firm": "company",
    "input_text": "text",
    "work_life_balance_label": "work_life_balance",
    "comp_benefits_label": "compensation_benefits",
    "senior_mgmt_label": "senior_management"
}

df_standard = df_train[
    list(column_mapping.keys()) +
    ["is_mixed", "label_combination"]
].rename(columns=column_mapping).copy()

display(df_standard.head())

,id,company,text,work_life_balance,compensation_benefits,senior_management,is_mixed,label_combination
0,198590,EY,Headline: Great place for Grads to learn and d...,Medium,High,Medium,False,Medium_High_Medium
1,65111,Babcock-International-Group,Headline: 17 years and still here\nPros: Great...,Low,Medium,Medium,False,Low_Medium_Medium
2,648340,next,"Headline: Great benefits, too rigid in terms o...",Low,High,Low,True,Low_High_Low
3,439020,Microsoft,Headline: Microsoft veteran - 10 years\nPros: ...,High,High,High,False,High_High_High
4,33142,Apple,Headline: Great culture\nPros: open-minded env...,High,High,High,False,High_High_High


In [6]:
ASPECTS = [
    "work_life_balance",
    "compensation_benefits",
    "senior_management"
]

LABELS = ["Low", "Medium", "High"]

In [14]:
# Reconstruct review text safely
df_train["input_text"] = (
    "Headline: " + df_train["headline"].fillna("") +
    "\nPros: " + df_train["pros"].fillna("") +
    "\nCons: " + df_train["cons"].fillna("")
)

In [17]:
column_mapping = {
    "review_id": "id",
    "firm": "company",
    "input_text": "text",
    "work_life_balance_label": "work_life_balance",
    "comp_benefits_label": "compensation_benefits",
    "senior_mgmt_label": "senior_management"
}

df_standard = df_train[
    list(column_mapping.keys()) + ["is_mixed", "label_combination"]
].rename(columns=column_mapping).copy()

display(df_standard.head())

,id,company,text,work_life_balance,compensation_benefits,senior_management,is_mixed,label_combination
0,198590,EY,Headline: Great place for Grads to learn and d...,Medium,High,Medium,False,Medium_High_Medium
1,65111,Babcock-International-Group,Headline: 17 years and still here\nPros: Great...,Low,Medium,Medium,False,Low_Medium_Medium
2,648340,next,"Headline: Great benefits, too rigid in terms o...",Low,High,Low,True,Low_High_Low
3,439020,Microsoft,Headline: Microsoft veteran - 10 years\nPros: ...,High,High,High,False,High_High_High
4,33142,Apple,Headline: Great culture\nPros: open-minded env...,High,High,High,False,High_High_High


In [18]:
assert df_standard["id"].is_unique
assert df_standard[ASPECTS].isin(LABELS).all().all()
assert df_standard["text"].notna().all()
assert df_standard["text"].is_unique

print("Dataset shape:", df_standard.shape)
print("Missing texts:", df_standard["text"].isna().sum())
print("All validation checks passed.")

Dataset shape: (387057, 8)
Missing texts: 0
All validation checks passed.


In [19]:
## 4. Stratified Training Data Sampling

##Select 20,000 real training samples while preserving
##the distribution of all 27 label combinations.

In [20]:
df_train_20k, _ = train_test_split(
    df_standard,
    train_size=20000,
    random_state=SEED,
    stratify=df_standard["label_combination"]
)

print("Training sample size:", len(df_train_20k))
print("Companies:", df_train_20k["company"].nunique())
print("Mixed reviews:", df_train_20k["is_mixed"].sum())
print("Mixed percentage:", round(df_train_20k["is_mixed"].mean() * 100, 2))

Training sample size: 20000
Companies: 335
Mixed reviews: 4482
Mixed percentage: 22.41


In [21]:
distribution_comparison = pd.DataFrame({
    "Original Train": df_standard["label_combination"]
        .value_counts(normalize=True) * 100,

    "Sample 20K": df_train_20k["label_combination"]
        .value_counts(normalize=True) * 100
}).round(2)

display(distribution_comparison)

,Original Train,Sample 20K
label_combination,,
High_High_High,25.11,25.11
Low_Low_Low,9.07,9.07
Medium_High_High,5.32,5.32
High_Medium_High,5.29,5.29
High_High_Medium,4.93,4.92
Medium_Medium_Medium,4.40,4.40
Low_Medium_Low,3.86,3.86
High_Medium_Medium,3.36,3.36
Medium_High_Medium,3.16,3.16


The stratified sampling successfully preserved the original
training distribution across all 27 label combinations.

The proportion of mixed reviews remained unchanged at 22.41%.
The resulting 20K dataset provides a representative training
sample in terms of joint label distribution.

In [ ]:
## 5. Nested Training Subsets

#Create stratified subsets of 10,000 and 2,000 reviews
#for hyperparameter tuning and low-resource experiments.

In [22]:
df_train_10k, _ = train_test_split(
    df_train_20k,
    train_size=10000,
    random_state=SEED,
    stratify=df_train_20k["label_combination"]
)

df_train_2k, _ = train_test_split(
    df_train_10k,
    train_size=2000,
    random_state=SEED,
    stratify=df_train_10k["label_combination"]
)

In [23]:
assert set(df_train_2k["id"]).issubset(df_train_10k["id"])
assert set(df_train_10k["id"]).issubset(df_train_20k["id"])

for name, subset in [
    ("20K", df_train_20k),
    ("10K", df_train_10k),
    ("2K", df_train_2k)
]:
    print(f"\nSubset: {name}")
    print("Samples:", len(subset))
    print("Companies:", subset["company"].nunique())
    print("Label combinations:", subset["label_combination"].nunique())
    print("Mixed reviews:", subset["is_mixed"].sum())
    print("Mixed percentage:", round(subset["is_mixed"].mean() * 100, 2))

print("\nNested subset verification passed.")


Subset: 20K
Samples: 20000
Companies: 335
Label combinations: 27
Mixed reviews: 4482
Mixed percentage: 22.41

Subset: 10K
Samples: 10000
Companies: 315
Label combinations: 27
Mixed reviews: 2240
Mixed percentage: 22.4

Subset: 2K
Samples: 2000
Companies: 218
Label combinations: 27
Mixed reviews: 448
Mixed percentage: 22.4

Nested subset verification passed.


In [ ]:
## 6. Prepare Evaluation Datasets

#Load and standardize the validation, test, and unseen
#company datasets for the experimental evaluation protocol.

In [24]:
def standardize_dataset(df):
    df = df.copy()

    df["input_text"] = (
        "Headline: " + df["headline"].fillna("") +
        "\nPros: " + df["pros"].fillna("") +
        "\nCons: " + df["cons"].fillna("")
    )

    return df[
        list(column_mapping.keys()) +
        ["is_mixed", "label_combination"]
    ].rename(columns=column_mapping)


df_val = standardize_dataset(
    pd.read_parquet(DATA_DIR / "validation.parquet")
)

df_test = standardize_dataset(
    pd.read_parquet(DATA_DIR / "test.parquet")
)

df_unseen = standardize_dataset(
    pd.read_parquet(DATA_DIR / "unseen.parquet")
)

In [25]:
for name, df in [
    ("Validation", df_val),
    ("Test", df_test),
    ("Unseen", df_unseen)
]:
    assert df["id"].is_unique
    assert df["text"].notna().all()
    assert df["text"].is_unique
    assert df[ASPECTS].isin(LABELS).all().all()

    print(f"{name}: {len(df):,} samples - verified")

Validation: 82,941 samples - verified
Test: 82,941 samples - verified
Unseen: 97,981 samples - verified


In [ ]:
## 7. Data Leakage Verification

#Verify that training and evaluation datasets do not
#share review IDs or identical review texts.

#Ensure that unseen companies are completely excluded
#from the training, validation, and standard test datasets.

In [26]:
from itertools import combinations

datasets = {
    "Train": df_standard,
    "Validation": df_val,
    "Test": df_test,
    "Unseen": df_unseen
}

for (name_a, df_a), (name_b, df_b) in combinations(
    datasets.items(), 2
):
    id_overlap = set(df_a["id"]) & set(df_b["id"])
    text_overlap = set(df_a["text"]) & set(df_b["text"])

    print(f"{name_a} vs {name_b}")
    print("ID overlap:", len(id_overlap))
    print("Text overlap:", len(text_overlap))
    print()

    assert len(id_overlap) == 0
    assert len(text_overlap) == 0

Train vs Validation
ID overlap: 0
Text overlap: 0

Train vs Test
ID overlap: 0
Text overlap: 0

Train vs Unseen
ID overlap: 0
Text overlap: 0

Validation vs Test
ID overlap: 0
Text overlap: 0

Validation vs Unseen
ID overlap: 0
Text overlap: 0

Test vs Unseen
ID overlap: 0
Text overlap: 0



In [27]:
unseen_companies = set(df_unseen["company"])

for name in ["Train", "Validation", "Test"]:
    overlap = unseen_companies & set(datasets[name]["company"])

    print(f"Unseen vs {name}: {len(overlap)} overlapping companies")

    assert len(overlap) == 0

print("\nAll data leakage checks passed.")

Unseen vs Train: 0 overlapping companies
Unseen vs Validation: 0 overlapping companies
Unseen vs Test: 0 overlapping companies

All data leakage checks passed.


In [ ]:
## 8. Evaluation Dataset Sampling

##Create fixed stratified evaluation datasets of 2,000
#reviews each for validation, clean testing, and unseen
#company evaluation.

#The mixed-review test set (T3) is derived from T1.

In [28]:
EVAL_SIZE = 2000

df_val_sample, _ = train_test_split(
    df_val,
    train_size=EVAL_SIZE,
    random_state=SEED,
    stratify=df_val["label_combination"]
)

df_t1, _ = train_test_split(
    df_test,
    train_size=EVAL_SIZE,
    random_state=SEED,
    stratify=df_test["label_combination"]
)

df_t3 = df_t1[df_t1["is_mixed"]].copy()

df_t4, _ = train_test_split(
    df_unseen,
    train_size=EVAL_SIZE,
    random_state=SEED,
    stratify=df_unseen["label_combination"]
)

In [29]:
evaluation_sets = {
    "Validation": df_val_sample,
    "T1 - Clean": df_t1,
    "T3 - Mixed": df_t3,
    "T4 - Unseen": df_t4
}

for name, df in evaluation_sets.items():
    print(f"\n{name}")
    print("Samples:", len(df))
    print("Companies:", df["company"].nunique())
    print("Label combinations:", df["label_combination"].nunique())
    print("Mixed reviews:", df["is_mixed"].sum())
    print("Mixed percentage:", round(df["is_mixed"].mean() * 100, 2))

assert len(df_val_sample) == EVAL_SIZE
assert len(df_t1) == EVAL_SIZE
assert len(df_t4) == EVAL_SIZE
assert set(df_t3["id"]).issubset(df_t1["id"])
assert df_t3["is_mixed"].all()

print("\nAll evaluation sampling checks passed.")


Validation
Samples: 2000
Companies: 215
Label combinations: 27
Mixed reviews: 449
Mixed percentage: 22.45

T1 - Clean
Samples: 2000
Companies: 226
Label combinations: 27
Mixed reviews: 449
Mixed percentage: 22.45

T3 - Mixed
Samples: 449
Companies: 110
Label combinations: 12
Mixed reviews: 449
Mixed percentage: 100.0

T4 - Unseen
Samples: 2000
Companies: 54
Label combinations: 27
Mixed reviews: 467
Mixed percentage: 23.35

All evaluation sampling checks passed.


In [ ]:
## 9. Define Text Noise Functions

#Implement reproducible character-level typos and
#punctuation noise for robustness evaluation and
#training data augmentation.

In [30]:
import random
import re


def add_typo_noise(text, rng, probability=0.05):
    def modify_word(match):
        word = match.group()

        if len(word) < 4 or rng.random() >= probability:
            return word

        chars = list(word)
        operation = rng.choice(["delete", "swap"])

        if operation == "delete":
            idx = rng.randrange(1, len(chars) - 1)
            del chars[idx]

        else:
            idx = rng.randrange(0, len(chars) - 1)
            chars[idx], chars[idx + 1] = chars[idx + 1], chars[idx]

        return "".join(chars)

    # Preserve the review structure and field names
    return re.sub(
        r"\b(?!Headline\b|Pros\b|Cons\b)[a-zA-Z]+\b",
        modify_word,
        text
    )


def add_punctuation_noise(text, rng, probability=0.3):
    return re.sub(
        r"[,.!?;]",
        lambda match: (
            "" if rng.random() < probability
            else match.group()
        ),
        text
    )


def add_combined_noise(text, rng):
    text = add_typo_noise(text, rng)
    text = add_punctuation_noise(text, rng)

    return text

In [31]:
example = df_t1.iloc[0]["text"]

print("Original:")
print(example)

print("\nTypo Noise:")
print(add_typo_noise(example, random.Random(SEED)))

print("\nPunctuation Noise:")
print(add_punctuation_noise(example, random.Random(SEED)))

print("\nCombined Noise:")
print(add_combined_noise(example, random.Random(SEED)))

Original:
Headline: A Fair Place to Work
Pros: I worked at this company for 15 years. The best thing is there stock options and 401k Plan is really great.
Cons: In my position it was much room for advancement in the work place.

Typo Noise:
Headline: A Fair Palce to Work
Pros: I worked at this company for 15 years. The best ting is there stok options and 401k Plan is really great.
Cons: In my position it was muh room for advancement in the work place.

Punctuation Noise:
Headline: A Fair Place to Work
Pros: I worked at this company for 15 years. The best thing is there stock options and 401k Plan is really great
Cons: In my position it was much room for advancement in the work place

Combined Noise:
Headline: A Fair Palce to Work
Pros: I worked at this company for 15 years The best ting is there stok options and 401k Plan is really great
Cons: In my position it was muh room for advancement in the work place.


In [32]:
## 10. Generate Noisy Evaluation Datasets
# Create reproducible noisy versions of T1.

df_t2_typo = df_t1.copy()
df_t2_punct = df_t1.copy()
df_t2 = df_t1.copy()

rng_typo = random.Random(SEED)
rng_punct = random.Random(SEED)

df_t2_typo["text"] = df_t1["text"].apply(
    lambda text: add_typo_noise(text, rng_typo)
)

df_t2_punct["text"] = df_t1["text"].apply(
    lambda text: add_punctuation_noise(text, rng_punct)
)

rng_combined = random.Random(SEED)

df_t2["text"] = df_t2_typo["text"].apply(
    lambda text: add_punctuation_noise(text, rng_combined)
)

In [33]:
## Verify Noisy Evaluation Datasets
# Ensure that only the review text was modified.

for name, df in [
    ("T2_typo", df_t2_typo),
    ("T2_punct", df_t2_punct),
    ("T2", df_t2)
]:
    assert len(df) == len(df_t1)
    assert df["id"].equals(df_t1["id"])
    assert df.drop(columns="text").equals(
        df_t1.drop(columns="text")
    )
    assert df["text"].notna().all()

    changed = (df["text"] != df_t1["text"]).sum()

    print(f"\n{name}")
    print("Total samples:", len(df))
    print("Modified reviews:", changed)
    print("Unmodified reviews:", len(df) - changed)
    print("Modified percentage:",
          round(changed / len(df) * 100, 2))

print("\nAll noisy dataset checks passed.")


T2_typo
Total samples: 2000
Modified reviews: 1236
Unmodified reviews: 764
Modified percentage: 61.8

T2_punct
Total samples: 2000
Modified reviews: 1086
Unmodified reviews: 914
Modified percentage: 54.3

T2
Total samples: 2000
Modified reviews: 1567
Unmodified reviews: 433
Modified percentage: 78.35

All noisy dataset checks passed.


In [34]:
## 11. Noise Augmentation (C2)
# Select stratified training samples for noise augmentation.

df_noise_source_10k = df_train_10k.copy()

df_noise_source_1k, _ = train_test_split(
    df_train_2k,
    train_size=1000,
    random_state=SEED,
    stratify=df_train_2k["label_combination"]
)

In [35]:
## Generate Noise-Augmented Training Samples

def create_noise_dataset(df, prefix):
    augmented = df.copy().reset_index(drop=True)
    rng = random.Random(SEED)

    augmented["source_id"] = augmented["id"]

    augmented["id"] = [
        f"{prefix}_{i}" for i in range(len(augmented))
    ]

    augmented["text"] = augmented["text"].apply(
        lambda text: add_combined_noise(text, rng)
    )

    return augmented


df_aug_noise_10k = create_noise_dataset(
    df_noise_source_10k, "noise_10k"
)

df_aug_noise_1k = create_noise_dataset(
    df_noise_source_1k, "noise_1k"
)

In [36]:
## Validate Noise-Augmented Training Samples

for name, augmented, source in [
    ("Full Training", df_aug_noise_10k, df_noise_source_10k),
    ("Low Resource", df_aug_noise_1k, df_noise_source_1k)
]:
    source = source.reset_index(drop=True)

    assert len(augmented) == len(source)
    assert augmented["id"].is_unique
    assert augmented["source_id"].equals(source["id"])
    assert augmented[ASPECTS].equals(source[ASPECTS])
    assert augmented["text"].notna().all()

    changed = (augmented["text"] != source["text"]).sum()

    print(f"\n{name}")
    print("Augmented samples:", len(augmented))
    print("Modified reviews:", changed)
    print("Modified percentage:",
          round(changed / len(augmented) * 100, 2))

assert set(df_aug_noise_1k["source_id"]).issubset(
    df_train_2k["id"]
)

print("\nAll noise augmentation checks passed.")


Full Training
Augmented samples: 10000
Modified reviews: 7834
Modified percentage: 78.34

Low Resource
Augmented samples: 1000
Modified reviews: 773
Modified percentage: 77.3

All noise augmentation checks passed.


In [37]:
## 11.1. Ensure Effective Noise Augmentation
# Retry noise generation for unchanged reviews.

def ensure_modified_reviews(augmented, source, max_attempts=100):
    augmented = augmented.copy().reset_index(drop=True)
    source = source.reset_index(drop=True)

    rng = random.Random(SEED + 1)

    for i in range(len(augmented)):
        original = source.at[i, "text"]

        attempts = 0

        while (
            augmented.at[i, "text"] == original
            and attempts < max_attempts
        ):
            augmented.at[i, "text"] = add_combined_noise(
                original, rng
            )

            attempts += 1

    return augmented


df_aug_noise_10k = ensure_modified_reviews(
    df_aug_noise_10k, df_noise_source_10k
)

df_aug_noise_1k = ensure_modified_reviews(
    df_aug_noise_1k, df_noise_source_1k
)

In [38]:
## Verify Effective Noise Augmentation

for name, augmented, source in [
    ("Full Training", df_aug_noise_10k, df_noise_source_10k),
    ("Low Resource", df_aug_noise_1k, df_noise_source_1k)
]:
    source = source.reset_index(drop=True)

    changed = (augmented["text"] != source["text"]).sum()

    assert len(augmented) == len(source)
    assert augmented["id"].is_unique
    assert augmented[ASPECTS].equals(source[ASPECTS])

    print(f"\n{name}")
    print("Total samples:", len(augmented))
    print("Modified reviews:", changed)
    print("Unmodified reviews:", len(augmented) - changed)
    print("Modified percentage:",
          round(changed / len(augmented) * 100, 2))


Full Training
Total samples: 10000
Modified reviews: 10000
Unmodified reviews: 0
Modified percentage: 100.0

Low Resource
Total samples: 1000
Modified reviews: 1000
Unmodified reviews: 0
Modified percentage: 100.0


In [39]:
## 12. Mixed Review Oversampling (C3)
# Oversample mixed reviews from the corresponding training sets.

def create_mixed_oversample(df, n_samples, prefix):
    mixed = df[df["is_mixed"]].copy()

    augmented = mixed.sample(
        n=n_samples,
        replace=True,
        random_state=SEED
    ).reset_index(drop=True)

    augmented["source_id"] = augmented["id"]

    augmented["id"] = [
        f"{prefix}_{i}" for i in range(len(augmented))
    ]

    return augmented


df_aug_mixed_10k = create_mixed_oversample(
    df_train_20k, 10000, "mixed_10k"
)

df_aug_mixed_1k = create_mixed_oversample(
    df_train_2k, 1000, "mixed_1k"
)

In [40]:
## Validate Mixed Review Oversampling

for name, augmented, source, expected_size in [
    ("Full Training", df_aug_mixed_10k, df_train_20k, 10000),
    ("Low Resource", df_aug_mixed_1k, df_train_2k, 1000)
]:
    assert len(augmented) == expected_size
    assert augmented["id"].is_unique
    assert augmented["is_mixed"].all()

    assert set(augmented["source_id"]).issubset(
        source["id"]
    )

    print(f"\n{name}")
    print("Augmented samples:", len(augmented))
    print("Unique source reviews:", augmented["source_id"].nunique())
    print("Label combinations:", augmented["label_combination"].nunique())
    print("Mixed percentage:", augmented["is_mixed"].mean() * 100)

print("\nAll mixed oversampling checks passed.")


Full Training
Augmented samples: 10000
Unique source reviews: 3980
Label combinations: 12
Mixed percentage: 100.0

Low Resource
Augmented samples: 1000
Unique source reviews: 395
Label combinations: 12
Mixed percentage: 100.0

All mixed oversampling checks passed.


In [41]:
## 13. Final Training Dataset Summary
# Summarize training sizes and mixed-review proportions
# for experimental conditions C1-C3.

training_summary = []

for scale, base, noise, mixed in [
    ("Full", df_train_20k, df_aug_noise_10k, df_aug_mixed_10k),
    ("Low Resource", df_train_2k, df_aug_noise_1k, df_aug_mixed_1k)
]:
    for condition, augmentation in [
        ("C1 - Real Only", None),
        ("C2 - Noise", noise),
        ("C3 - Mixed Oversampling", mixed)
    ]:
        combined = (
            base if augmentation is None
            else pd.concat([base, augmentation], ignore_index=True)
        )

        training_summary.append({
            "Scale": scale,
            "Condition": condition,
            "Real Samples": len(base),
            "Augmented Samples": (
                0 if augmentation is None else len(augmentation)
            ),
            "Total Samples": len(combined),
            "Mixed Reviews": combined["is_mixed"].sum(),
            "Mixed Percentage": round(
                combined["is_mixed"].mean() * 100, 2
            )
        })

df_training_summary = pd.DataFrame(training_summary)

display(df_training_summary)

,Scale,Condition,Real Samples,Augmented Samples,Total Samples,Mixed Reviews,Mixed Percentage
0,Full,C1 - Real Only,20000,0,20000,4482,22.41
1,Full,C2 - Noise,20000,10000,30000,6722,22.41
2,Full,C3 - Mixed Oversampling,20000,10000,30000,14482,48.27
3,Low Resource,C1 - Real Only,2000,0,2000,448,22.40
4,Low Resource,C2 - Noise,2000,1000,3000,674,22.47
5,Low Resource,C3 - Mixed Oversampling,2000,1000,3000,1448,48.27


In [42]:
## 14. Verify Updated Source Datasets
# Ensure that the corrected Parquet files match
# the datasets already prepared in this notebook.

for name, filename, previous_df in [
    ("Train", "train.parquet", df_standard),
    ("Validation", "validation.parquet", df_val),
    ("Test", "test.parquet", df_test),
    ("Unseen", "unseen.parquet", df_unseen)
]:
    updated_raw = pd.read_parquet(DATA_DIR / filename)

    assert updated_raw["input_text"].notna().all()

    updated_df = standardize_dataset(updated_raw)

    pd.testing.assert_frame_equal(
        updated_df.sort_values("id").reset_index(drop=True),
        previous_df.sort_values("id").reset_index(drop=True)
    )

    print(f"{name}: identical and verified")

print("\nAll updated datasets verified successfully.")

Train: identical and verified
Validation: identical and verified
Test: identical and verified
Unseen: identical and verified

All updated datasets verified successfully.


In [43]:
## 15. Export Experimental Datasets
# Create a dedicated directory for experimental datasets.

PROJECT_DIR = Path("/content/drive/MyDrive/LLM Project")

EXPORT_DIR = PROJECT_DIR / "DATASETS" / "EXPERIMENTS" / "v1"

assert PROJECT_DIR.is_dir()

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Export directory:", EXPORT_DIR)

Export directory: /content/drive/MyDrive/LLM Project/DATASETS/EXPERIMENTS/v1


In [44]:
## Export Training, Evaluation and Augmentation Datasets

export_datasets = {
    "train": df_train_20k,
    "train_10k": df_train_10k,
    "train_2k": df_train_2k,
    "val": df_val_sample,
    "T1": df_t1,
    "T2": df_t2,
    "T2_typo": df_t2_typo,
    "T2_punct": df_t2_punct,
    "T3": df_t3,
    "T4": df_t4,
    "aug_noise": df_aug_noise_10k,
    "aug_noise_1k": df_aug_noise_1k,
    "aug_oversample_mixed": df_aug_mixed_10k,
    "aug_oversample_mixed_1k": df_aug_mixed_1k
}

EXPORT_COLUMNS = [
    "id",
    "company",
    "text",
    "work_life_balance",
    "compensation_benefits",
    "senior_management"
]

for name, df in export_datasets.items():
    columns = EXPORT_COLUMNS.copy()

    if "source_id" in df.columns:
        columns.append("source_id")

    path = EXPORT_DIR / f"{name}.jsonl"

    df[columns].to_json(
        path,
        orient="records",
        lines=True,
        force_ascii=False
    )

    print(f"Saved {name}: {len(df):,} samples")

df_training_summary.to_csv(
    EXPORT_DIR / "training_summary.csv",
    index=False
)

print("\nAll experimental datasets exported.")

Saved train: 20,000 samples
Saved train_10k: 10,000 samples
Saved train_2k: 2,000 samples
Saved val: 2,000 samples
Saved T1: 2,000 samples
Saved T2: 2,000 samples
Saved T2_typo: 2,000 samples
Saved T2_punct: 2,000 samples
Saved T3: 449 samples
Saved T4: 2,000 samples
Saved aug_noise: 10,000 samples
Saved aug_noise_1k: 1,000 samples
Saved aug_oversample_mixed: 10,000 samples
Saved aug_oversample_mixed_1k: 1,000 samples

All experimental datasets exported.


In [45]:
## 16. Verify Exported Datasets
# Reload exported files and verify their integrity.

for name, original_df in export_datasets.items():

    path = EXPORT_DIR / f"{name}.jsonl"

    assert path.is_file()

    loaded_df = pd.read_json(
        path,
        lines=True,
        convert_dates=False
    )

    columns = EXPORT_COLUMNS.copy()

    if "source_id" in original_df.columns:
        columns.append("source_id")

    pd.testing.assert_frame_equal(
        loaded_df[columns].reset_index(drop=True),
        original_df[columns].reset_index(drop=True),
        check_dtype=False
    )

    assert loaded_df["id"].is_unique
    assert loaded_df["text"].notna().all()
    assert loaded_df[ASPECTS].isin(LABELS).all().all()

    print(f"{name}: {len(loaded_df):,} samples - verified")

print("\nAll exported datasets verified successfully.")

train: 20,000 samples - verified
train_10k: 10,000 samples - verified
train_2k: 2,000 samples - verified
val: 2,000 samples - verified
T1: 2,000 samples - verified
T2: 2,000 samples - verified
T2_typo: 2,000 samples - verified
T2_punct: 2,000 samples - verified
T3: 449 samples - verified
T4: 2,000 samples - verified
aug_noise: 10,000 samples - verified
aug_noise_1k: 1,000 samples - verified
aug_oversample_mixed: 10,000 samples - verified
aug_oversample_mixed_1k: 1,000 samples - verified

All exported datasets verified successfully.
